In [ ]:
import os
import sys
import logging
from datetime import datetime  # 1. 导入 datetime 模块
import pandas as pd

from make_ids_map_table_utils import *

In [ ]:
# --- 配置 ---
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

PROXY_URL = "http://127.0.0.1:7897"
os.environ["http_proxy"] = PROXY_URL
os.environ["https_proxy"] = PROXY_URL

class Config:
    TIMESTAMP = datetime.now().strftime('%y%m%d') 

    IN_DIR = '../data/input'
    IN_OUT_DIR = '../data/in_out'
    OUT_DIR = '../data/output'
    TEMP_DIR = '../tmp_data'

    MISSING_DATA_MAG_DIR = os.path.join(TEMP_DIR, 'mis_data_mag')
    DEDUPLICATE_DIR = os.path.join(TEMP_DIR, 'deduplicate')
    
    # 输入文件路径
    RAW_DATASET = os.path.join(IN_DIR, '0_raw-dataset.csv')
 
    # 输出文件路径
    IDS_MAP_TABLE = os.path.join(IN_OUT_DIR, f'{TIMESTAMP}_raw-ds_fetch-ids_add-ids.csv')
    IDS_MAP_TABLE_OUT = os.path.join(OUT_DIR, f'{TIMESTAMP}_out_ids-map-tb.csv')
    
    BYPRODUCT_TABLE = os.path.join(IN_OUT_DIR, f'{TIMESTAMP}_ids-map-tb_dd_fg.csv')
    BYPRODUCT_TABLE_OUT = os.path.join(OUT_DIR, f'{TIMESTAMP}_out_byproduct-tb.csv')

In [ ]:
# 检查输入文件
if not os.path.exists(Config.RAW_DATASET):
    logging.error(f"File not found: {Config.RAW_DATASET}")
    # 脚本模式下使用 sys.exit 或直接停止
    sys.exit(1)

logging.info("Loading dataset...")

# 2. 自动识别后缀并读取
file_ext = os.path.splitext(Config.RAW_DATASET)[1].lower()

try:
    if file_ext == '.csv':
        raw_dataset = pd.read_csv(Config.RAW_DATASET, encoding='utf-8-sig') 
    elif file_ext in ['.xlsx', '.xls']:
        raw_dataset = pd.read_excel(Config.RAW_DATASET)
except Exception as e:
    print(f"读取文件出错: {e}")

# 提取唯一化合物 (去重且清理无效值)
# 这一步将 c1, c2, c3 三列展平并提取唯一值
compound_array = pd.unique(raw_dataset[['c1', 'c2', 'c3']].values.ravel())
compounds_table = pd.DataFrame(compound_array, columns=['compound']).dropna()
compounds_table = compounds_table[compounds_table['compound'].astype(str).str.strip() != '']

logging.info(f"Processing {len(compounds_table)} compounds...")

# 调用从 utils 导入的函数
# 注意：apply 返回的是 tuple 列表，后续需要拆分
results = compounds_table['compound'].apply(f_fetch_compound_info)

# 结果整合与拆分
# 将 apply 返回的 (smiles, name, cas) 元组拆分为三列
result_df = pd.DataFrame(
    results.tolist(), 
    index=compounds_table.index, 
    columns=['ismiles', 'uname', 'cas']
)
ids_map_table = pd.concat([compounds_table, result_df], axis=1)

# 保存结果
ids_map_table.to_csv(Config.IDS_MAP_TABLE, index = False)
ids_map_table.to_csv(Config.IDS_MAP_TABLE_OUT, index = False)

人工检查数据集质量，填充空值

In [ ]:
qlty_sum_output_path = os.path.join(Config.MISSING_DATA_MAG_DIR, 'ids-map-tb_qlty-sum.xlsx')
data_quality_summary(ids_map_table, qlty_sum_output_path)

In [ ]:
ids_map_table = pd.read_csv(Config.IDS_MAP_TABLE)
ids_map_table = ids_map_table.dropna()

In [ ]:
group_cols = ['ismiles', 'uname', 'cas']
byproduct_table = deduplicate_byproduct_table(ids_map_table, group_cols, Config.DEDUPLICATE_DIR )
byproduct_table.to_csv(Config.BYPRODUCT_TABLE , index=False)
byproduct_table.to_csv(Config.BYPRODUCT_TABLE_OUT , index=False)